# Farm Cartoon LoRA Training
**Runtime > Change runtime type > A100 GPU**

This notebook trains a LoRA on your cartoon sprites so the model learns your exact art style.
Once trained, you can generate unlimited new sprites and backgrounds in your style.

In [ ]:
# Step 1 — Install dependencies
!pip install -q diffusers transformers accelerate peft torch huggingface_hub bitsandbytes torchvision
!pip install -q prodigyopt
print('Done!')

In [ ]:
# Step 2 — Login to HuggingFace
from huggingface_hub import login
login()

In [ ]:
# Step 3 — Download the FLUX training script from diffusers
!wget -q https://raw.githubusercontent.com/huggingface/diffusers/main/examples/dreambooth/train_dreambooth_lora_flux.py
print('Training script ready!')

In [ ]:
# Step 4 — Upload your sprites as training data
# Upload all your best sprite PNGs — aim for 20-50 images
# These should all share the same cartoon style
import os
from google.colab import files

os.makedirs('training_data', exist_ok=True)
uploaded = files.upload()

# Move uploaded files into training_data folder
for filename in uploaded.keys():
    os.rename(filename, f'training_data/{filename}')

print(f'Uploaded {len(uploaded)} files to training_data/')

In [ ]:
# Step 5 — Create captions for each training image
# Each image needs a text description so the model learns what to associate with your style
import os
from PIL import Image

training_dir = 'training_data'
image_files = [f for f in os.listdir(training_dir) if f.endswith('.png') or f.endswith('.jpg')]

# The trigger word — use this in prompts later to activate your style
TRIGGER_WORD = 'frmcrtoon'

for img_file in image_files:
    name = os.path.splitext(img_file)[0]
    parts = name.split('_')

    # Build a caption from the filename e.g. dog_happy -> cartoon dog happy pose
    if len(parts) >= 2:
        animal = parts[0]
        action = '_'.join(parts[1:])
        caption = f'{TRIGGER_WORD} style cartoon {animal} {action} pose, flat design, thick outlines, white background'
    else:
        caption = f'{TRIGGER_WORD} style cartoon character, flat design, thick outlines, white background'

    # Save caption as a .txt file with the same name
    caption_path = os.path.join(training_dir, f'{name}.txt')
    with open(caption_path, 'w') as f:
        f.write(caption)
    print(f'{img_file} -> {caption}')

print(f'\nCreated captions for {len(image_files)} images')

In [ ]:
# Step 6 — Train the LoRA
# This takes about 20-40 minutes on an A100
!accelerate launch train_dreambooth_lora_flux.py \
  --pretrained_model_name_or_path='black-forest-labs/FLUX.1-dev' \
  --instance_data_dir='training_data' \
  --output_dir='lora_output' \
  --mixed_precision='bf16' \
  --instance_prompt='frmcrtoon style cartoon character' \
  --resolution=512 \
  --train_batch_size=1 \
  --gradient_accumulation_steps=4 \
  --learning_rate=1e-4 \
  --lr_scheduler='constant' \
  --lr_warmup_steps=0 \
  --max_train_steps=1000 \
  --rank=16 \
  --seed=42 \
  --checkpointing_steps=500

In [ ]:
# Step 7 — Test your trained LoRA
import torch
from diffusers import FluxPipeline

pipe = FluxPipeline.from_pretrained(
    'black-forest-labs/FLUX.1-dev',
    torch_dtype=torch.bfloat16
)
pipe.load_lora_weights('lora_output')
pipe.enable_model_cpu_offload()

# Use the trigger word in your prompt
prompt = 'frmcrtoon style cartoon dog jumping in the air, flat design, thick outlines, white background'

result = pipe(prompt=prompt, num_inference_steps=28, guidance_scale=3.5).images[0]
result.save('test_lora_output.png')

from IPython.display import display
display(result)
print('Done!')

In [ ]:
# Step 8 — Download your trained LoRA weights
# This is YOUR model — save it to your computer
import zipfile
from google.colab import files

with zipfile.ZipFile('my_cartoon_lora.zip', 'w') as z:
    for f in os.listdir('lora_output'):
        z.write(f'lora_output/{f}')

files.download('my_cartoon_lora.zip')
print('LoRA weights downloaded!')

In [ ]:
# BONUS — Generate a full batch of new sprites with your trained style
SPRITES_TO_GENERATE = [
    ('dog_running_new.png',   'frmcrtoon style cartoon dog running fast, flat design, thick outlines, white background'),
    ('dog_sleeping_new.png',  'frmcrtoon style cartoon dog sleeping curled up, flat design, thick outlines, white background'),
    ('pig_dancing_new.png',   'frmcrtoon style cartoon pig dancing arms up, flat design, thick outlines, white background'),
    ('cow_eating_new.png',    'frmcrtoon style cartoon cow eating grass, flat design, thick outlines, white background'),
    ('lamb_jumping_new.png',  'frmcrtoon style cartoon lamb jumping, flat design, thick outlines, white background'),
]

for output_file, prompt in SPRITES_TO_GENERATE:
    print(f'Generating {output_file}...')
    result = pipe(prompt=prompt, num_inference_steps=28, guidance_scale=3.5).images[0]
    result.save(output_file)
    print(f'  Saved!')

# Download all as zip
with zipfile.ZipFile('new_sprites.zip', 'w') as z:
    for output_file, _ in SPRITES_TO_GENERATE:
        if os.path.exists(output_file):
            z.write(output_file)

files.download('new_sprites.zip')
print('All sprites downloaded!')